# Development, Differentiation, and the Making of Organisms Workflow

This notebook scaffold supports the article **Development, Differentiation, and the Making of Organisms**. It can be expanded with growth modeling, differentiation dynamics, morphogen gradients, reaction-diffusion patterning, state transitions, condition scoring, and provenance notes.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

article_dir = Path.cwd().parent
growth = pd.read_csv(article_dir / 'data' / 'developmental_growth.csv')
early = growth.iloc[:5]
slope, intercept = np.polyfit(early['time_h'], np.log(early['cells']), 1)
r_est = slope
doubling_time_h = np.log(2) / r_est
pd.DataFrame({'r_est':[r_est], 'doubling_time_h':[doubling_time_h]}).round(4)

In [ ]:
lineage = pd.read_csv(article_dir / 'data' / 'lineage_scenarios.csv')
lineage['total_commitment_rate'] = lineage['k1'] + lineage['k2']
lineage['lineage_1_fraction'] = lineage['k1'] / (lineage['k1'] + lineage['k2'])
lineage.round(4)

In [ ]:
morph = pd.read_csv(article_dir / 'data' / 'morphogen_gradient.csv')
morph['fate'] = np.where(morph['morphogen'] > 0.60, 'fate_A', np.where(morph['morphogen'] > 0.25, 'fate_B', 'fate_C'))
morph.groupby('fate').agg(n_positions=('position', 'size'), min_position=('position', 'min'), max_position=('position', 'max')).reset_index()

In [ ]:
state_df = pd.read_csv(article_dir / 'data' / 'state_transition_matrix.csv')
states = state_df['state'].tolist()
P = state_df[states].to_numpy(float)
x = np.array([0.90, 0.08, 0.01, 0.01])
trajectory = [x]
for _ in range(20):
    x = x @ P
    trajectory.append(x.copy())
pd.DataFrame(trajectory, columns=states).round(4).tail()

In [ ]:
condition = pd.read_csv(article_dir / 'data' / 'developmental_condition_sites.csv')
condition['developmental_condition_score'] = (
    0.18 * condition['growth_coherence'] +
    0.18 * condition['differentiation_signal'] +
    0.16 * condition['patterning_signal'] +
    0.16 * condition['morphogenesis_quality'] +
    0.16 * condition['environmental_stability'] +
    0.16 * (1 - condition['perturbation_risk'])
)
condition.sort_values('developmental_condition_score', ascending=False).round(3)